<p class="lead">A model that predicts well answers one question; the chemist's next question is always <em>why</em>. This unit builds the apparatus that answers it, and the controls that stop it from inventing answers. You will compute every explanation the field asks for — coefficients, impurity and permutation importance, partial dependence, ICE and ALE curves, Shapley values (derived from the axioms, computed exactly by enumeration and then approximated by Monte-Carlo sampling), LIME written by hand, counterfactuals and fingerprint-bit attributions — and then run the experiments that show how each of them fails: importance hidden by correlated features, partial dependence evaluated on molecules that cannot exist, a local surrogate that reproduces its black box only moderately well and disagrees with Shapley about what matters, and, most instructive of all, a <strong>model trained on scrambled labels whose explanation still looks chemically sensible</strong>.</p>

<div class="box"><h4>Resumen en español</h4>
<p>Esta unidad trata la interpretabilidad de los modelos. Se distinguen los modelos intrínsecamente interpretables de las explicaciones posteriores, y las explicaciones globales de las locales. Se calculan e interpretan los coeficientes de un modelo lineal regularizado, la importancia por impureza (y su sesgo hacia variables de alta cardinalidad), la importancia por permutación (incluida la permutación por grupos cuando las variables están correlacionadas), las curvas de dependencia parcial, ICE y ALE (con <code>iml</code> y también escritas a mano), los valores de Shapley (derivados de sus axiomas, calculados de forma exacta por enumeración y comparados con la aproximación de Monte-Carlo de <code>iml::Shapley</code> y con la descomposición de <code>DALEX</code>), LIME implementado a mano, las explicaciones contrafactuales y la atribución a nivel de bits de la huella molecular. Finalmente se aplican los controles de sanidad: explicar un modelo entrenado con etiquetas permutadas, medir la estabilidad de las explicaciones entre pliegues de validación cruzada, el efecto Rashomon y la objeción de Rudin.</p></div>

<div class="box key"><h4>Learning objectives</h4>
<ul class="objectives">
<li>Distinguish intrinsically interpretable models from post-hoc explanations, and global from local explanations.</li>
<li>Compute and read permutation importance, PDP/ICE/ALE, Shapley values and LIME, and state the failure mode of each: correlated features, extrapolation, instability, approximation error.</li>
<li>Map explanations back to the domain — fingerprint bits and substructures for molecules — and state what the available data does and does not allow.</li>
<li>Run the sanity checks that separate an explanation from a decoration: scrambled-label controls, stability across folds, and agreement between methods.</li>
<li>Write the interpretation section of a paper without over-claiming causality.</li>
</ul></div>

<div class="box bridge"><h4>How to read this unit</h4>
<p>Run every chunk; each number in the prose comes from the output above it. In a fresh Google Colab R runtime run <code>install.packages(c("data.table","glmnet","xgboost","ranger","iml","DALEX","patchwork"))</code> once before the first chunk. The models explained here are refitted inside this document in under a minute, so it is self-contained.</p></div>



In [ ]:
suppressPackageStartupMessages({
  library(data.table); library(Matrix); library(glmnet); library(xgboost)
  library(ranger);     library(iml);    library(DALEX);  library(ggplot2); library(patchwork)
})
set.seed(0)
PAL <- c(blue = "#2b5d9b", orange = "#d2691e", green = "#3a7d44", purple = "#6a4c93")
theme_set(theme_minimal(base_size = 9) +
          theme(plot.title = element_text(face = "bold", size = 10), panel.grid.minor = element_blank()))
cat("R", paste(R.version$major, R.version$minor, sep = "."),
    "| glmnet", as.character(packageVersion("glmnet")),
    "| xgboost", as.character(packageVersion("xgboost")),
    "| ranger",  as.character(packageVersion("ranger")),
    "| iml",     as.character(packageVersion("iml")),
    "| DALEX",   as.character(packageVersion("DALEX")), "\n")


In [ ]:
REPO <- "DanielBustosG/DanielBustosG.github.io"   # GitHub repository hosting this course (set once in common/config.json)
DATA_URL <- paste0("https://raw.githubusercontent.com/", REPO, "/main/DataScience/data/")
data_file <- function(name) {   # local ../data/<name>; in Colab, downloaded once into data/
  local <- file.path("..", "data", name)
  if (file.exists(local)) return(local)
  dest <- file.path("data", name)
  if (!file.exists(dest)) {
    dir.create("data", showWarnings = FALSE)
    download.file(paste0(DATA_URL, name), dest, quiet = TRUE, mode = "wb")
  }
  dest
}
dp <- function(f) data_file(f)
df  <- fread(dp("rock1_chembl_curated.csv")); cmp <- df[censored == FALSE]
DESC <- c("MolWt","MolLogP","TPSA","HBD","HBA","RotB",
          "AromaticRings","RingCount","HeavyAtoms","FractionCSP3","QED")
Xd <- as.matrix(cmp[, ..DESC]); y <- cmp$pActivity; ycls <- as.integer(y >= 7)   # potent at 100 nM
sp <- fread(dp("rock1_morgan_sparse.csv.gz")); keep <- sp[id %in% cmp$inchikey]
FP <- sparseMatrix(i = match(keep$id, cmp$inchikey), j = keep$bit + 1L, x = 1, dims = c(nrow(cmp), 2048))
Xfp <- as.matrix(FP); colnames(Xfp) <- paste0("b", 0:2047); storage.mode(Xfp) <- "double"
set.seed(0)                                                    # stratified 75/25 split
ite <- sort(c(sample(which(ycls == 1), round(0.25*sum(ycls == 1))),
              sample(which(ycls == 0), round(0.25*sum(ycls == 0)))))
itr <- setdiff(seq_len(nrow(Xd)), ite)
mu <- colMeans(Xd[itr, ]); sg <- apply(Xd[itr, ], 2, sd); Zd <- scale(Xd, center = mu, scale = sg)
r2  <- function(a, p) 1 - sum((a - p)^2)/sum((a - mean(a))^2)
mae <- function(a, p) mean(abs(a - p))
auc <- function(lab, sc) { r <- rank(sc); n1 <- sum(lab == 1); n0 <- sum(lab == 0)
                           (sum(r[lab == 1]) - n1*(n1 + 1)/2)/(n1*n0) }
cat(sprintf("compounds %d (train %d / test %d), potent fraction %.1f%%\n",
            length(y), length(itr), length(ite), 100*mean(ycls)))


In [ ]:
set.seed(0); cvr <- cv.glmnet(Zd[itr, ], y[itr], alpha = 0, nfolds = 10)     # ridge
ridge_pred <- function(Z) as.numeric(predict(cvr, newx = Z, s = "lambda.min"))
PAR <- list(eta = 0.05, max_depth = 5, subsample = 0.9, colsample_bytree = 0.9,
            objective = "reg:squarederror", nthread = 4)
set.seed(0); gbm <- xgb.train(PAR, xgb.DMatrix(Xd[itr, ], label = y[itr]), nrounds = 400, verbose = 0)
gp <- function(X) predict(gbm, xgb.DMatrix(as.matrix(X)))
set.seed(0); rf_fp <- ranger(x = Xfp[itr, ], y = factor(ycls[itr]), num.trees = 300,
                             min.node.size = 2, probability = TRUE, seed = 1, num.threads = 4)
cat(sprintf("ridge on 11 descriptors        : test R2 %.3f  MAE %.3f log units\n",
            r2(y[ite], ridge_pred(Zd[ite, ])), mae(y[ite], ridge_pred(Zd[ite, ]))))
cat(sprintf("gradient boosting, descriptors : test R2 %.3f  MAE %.3f log units\n",
            r2(y[ite], gp(Xd[ite, ])), mae(y[ite], gp(Xd[ite, ]))))
cat(sprintf("random forest, 2048-bit Morgan : test ROC-AUC %.3f (classification, pActivity >= 7)\n",
            auc(ycls[ite], predict(rf_fp, data = Xfp[ite, ])$predictions[, 2])))



## 1. What an explanation is for

Three models were just fitted on the same 2,764 ROCK1 compounds<sup class="c"><a href="#r24">[24]</a></sup>, and they are explainable in three different senses. The ridge regression *is* its own explanation: eleven numbers, one per descriptor, and the model is nothing else. The gradient-boosting model is a sum of four hundred small `xgboost` trees<sup class="c"><a href="#r33">[33]</a></sup> — accurate on this table, but no human reads four hundred trees — so any statement about what it learned must be *reconstructed* after the fact. The fingerprint random forest is worse still: its features are 2,048 hashed bits with no chemical names.

That gives the two axes of this unit.

**Intrinsic versus post-hoc.** An intrinsically interpretable model is one whose structure is the explanation — linear models, single shallow trees, rule lists, generalised additive models. A post-hoc explanation is a second, simpler model of the first one's behaviour. Rudin's objection<sup class="c"><a href="#r1">[1]</a></sup> is that for high-stakes decisions this is the wrong trade: an explanation that is only an approximation of the model can be wrong in exactly the cases that matter, and in many tabular problems an interpretable model loses little or no accuracy. On *this* table the trade is real rather than rhetorical — the printed test \(R^2\) of the boosted model is nearly three times that of ridge regression — so the black box earns its place, and the controls of Section 10 are the price of using it<sup class="c"><a href="#r2">[2]</a></sup>.

**Global versus local.** A *global* explanation describes the model's average behaviour over a population ("across the test set, the number of hydrogen-bond acceptors matters most"). A *local* explanation describes one prediction ("this compound was predicted potent mainly because of its eight rings"). The two can disagree without either being wrong, and conflating them is the commonest error in interpretation sections.

A third distinction matters for chemistry in particular: an explanation is a statement about **the model**, not about **the molecule**. "The model used this feature" is a fact you can verify; "this feature causes potency" is a causal claim the model cannot support<sup class="c"><a href="#r16">[16]</a></sup>. Section 11 returns to this.

<div class="box bridge"><h4>Domain context — structure-activity relationships and what chemists want from a model</h4>
<p>A medicinal-chemistry project is a sequence of hypotheses of the form "adding this group here should raise potency". The accumulated set of such statements for a series is its <strong>structure-activity relationship</strong> (SAR). Our target, ROCK1, is a protein kinase: its inhibitors almost all place a hydrogen-bond donor/acceptor pair against the ATP-site <em>hinge</em> (fasudil's isoquinoline, Y-27632's aminopyridine) and a basic amine towards the solvent-exposed acidic residues<sup class="c"><a href="#r22">[22]</a></sup>. A model whose explanation highlights exactly those motifs has at least learned the known SAR; a model that highlights a solubilising tail has probably learned which laboratory published which series. Chemists also know that SAR is not smooth: <strong>activity cliffs</strong> — pairs of near-identical molecules with hundred-fold different potency<sup class="c"><a href="#r19">[19]</a><a href="#r20">[20]</a></sup> — are exactly where similarity-based models fail and where their explanations are least trustworthy.</p></div>

## 2. The baseline explanation: coefficients

Before any machine-learning explanation, fit the linear model and read it. A standardised coefficient \(\beta_j\) says: holding the other ten descriptors fixed, a one-standard-deviation increase in descriptor \(j\) changes the predicted pActivity by \(\beta_j\) log units. For a logistic model the same coefficient exponentiates to an odds ratio. This is the only explanation in the unit that is exact, because it *is* the model. Both are fitted here with `glmnet`<sup class="c"><a href="#r31">[31]</a></sup> at the ridge penalty chosen by ten-fold cross-validation.



In [ ]:
set.seed(0); lg <- cv.glmnet(Zd[itr, ], ycls[itr], family = "binomial", alpha = 0, nfolds = 10)
cf  <- as.numeric(coef(cvr, s = "lambda.min"))[-1]
lcf <- as.numeric(coef(lg,  s = "lambda.min"))[-1]
co <- data.frame(descriptor = DESC, ridge_beta_per_SD = cf, logistic_beta = lcf,
                 odds_ratio_per_SD = exp(lcf))
knitr::kable(co[order(-abs(co$ridge_beta_per_SD)), ], digits = 3, row.names = FALSE,
             caption = "Ridge coefficients and logistic odds ratios on standardised descriptors")
cm <- cor(Xd); diag(cm) <- 0
cat(sprintf("ridge intercept %.2f (training mean pActivity %.2f); logistic test ROC-AUC %.3f\n",
            as.numeric(coef(cvr, s = "lambda.min"))[1], mean(y[itr]),
            auc(ycls[ite], as.numeric(predict(lg, newx = Zd[ite, ], s = "lambda.min")))))
cat(sprintf("largest pairwise |correlation| among descriptors: %.2f (%s)\n", max(abs(cm)),
            paste(DESC[which(abs(cm) == max(abs(cm)), arr.ind = TRUE)[1, ]], collapse = " vs ")))



The signs are readable and, for a kinase series, broadly sensible: more rotatable bonds and higher drug-likeness go with higher measured potency, more hydrogen-bond acceptors and higher lipophilicity with lower potency. But the last printed line is the warning that governs the whole unit: two descriptors in this table correlate at 0.98 (molecular weight and heavy-atom count are almost the same measurement). When two columns carry the same information, the regression must split the credit between them, and the split is determined by noise and by the regularisation strength — not by chemistry. Every importance method in this unit inherits that problem in its own way.

## 3. Impurity importance and its bias

A tree ensemble reports a free importance: for each feature, the total reduction in the splitting criterion it produced, summed over all nodes and trees<sup class="c"><a href="#r3">[3]</a></sup>. It costs nothing because it is a by-product of training — `ranger(..., importance = "impurity")`<sup class="c"><a href="#r30">[30]</a></sup> returns it. It is also biased towards features with many possible split points: continuous and high-cardinality variables beat binary ones even when both are pure noise<sup class="c"><a href="#r4">[4]</a></sup>. The demonstration is simple: add two columns of pure noise, one continuous and one binary, and look at where they rank.



In [ ]:
set.seed(0)
Xn <- cbind(Xd, noise_continuous = rnorm(nrow(Xd)), noise_binary = sample(0:1, nrow(Xd), TRUE))
NAMES <- colnames(Xn)
set.seed(0); rfn <- ranger(x = Xn[itr, ], y = y[itr], num.trees = 300, min.node.size = 2,
                           importance = "impurity", seed = 1, num.threads = 4)
  # NOTE: ranger's seed = 0 means "ignore the R seed" and is NOT reproducible; use a non-zero seed
imp <- sort(rfn$variable.importance, decreasing = TRUE)
knitr::kable(data.frame(feature = names(imp), share_of_total = round(imp/sum(imp), 4)),
             row.names = FALSE, caption = "Impurity importance with two pure-noise columns added")
cat(sprintf("continuous noise ranks %d of %d; binary noise ranks %d of %d\n",
            which(names(imp) == "noise_continuous"), length(imp),
            which(names(imp) == "noise_binary"), length(imp)))
cat(sprintf("the continuous noise column is credited with %.1f%% of the total importance, although it is independent of the target by construction; the binary noise column gets %.1f%%\n",
            100*imp["noise_continuous"]/sum(imp), 100*imp["noise_binary"]/sum(imp)))



The printed ranking is the whole argument: a column of standard normal noise comes **sixth of thirteen**, above six of the eleven real descriptors and with 9.7 % of the total importance, while the random *binary* column comes last with 1.7 %. Both are equally uninformative; the difference is only that a continuous variable offers the tree thousands of candidate thresholds, and with enough thresholds some always fit the training residuals. **Never publish impurity importances.** They are a diagnostic of how the trees were built, not of what matters.

## 4. Permutation importance, and what correlation does to it

The model-agnostic replacement measures the *loss of performance* when one feature is destroyed: shuffle column \(j\) in the held-out data, re-score, and record the drop<sup class="c"><a href="#r3">[3]</a></sup>. It needs no refitting, it works for any model, and it is measured on data the model has not seen, so it reports predictive value rather than fitting behaviour. `iml::FeatureImp`<sup class="c"><a href="#r29">[29]</a></sup> implements it; it is six lines by hand, written out below so that the grouped variant can be added.

Its failure mode is correlation, in two forms. First, **masking**: if two columns carry the same information, destroying one leaves the model able to read the other, and both appear unimportant. Second, **extrapolation**: shuffling one column while the others keep their values creates feature combinations that cannot exist (a molecule with the molecular weight of a peptide and the heavy-atom count of benzene), so the model is scored on inputs from outside the data distribution — Hooker and colleagues' objection to the whole method<sup class="c"><a href="#r5">[5]</a></sup>. The repair for masking is to permute correlated features **as a group**.



In [ ]:
perm_imp <- function(predfun, X, yy, nrep = 20, seed = 0) {
  set.seed(seed); base <- r2(yy, predfun(X))
  t(sapply(seq_len(ncol(X)), function(j) {
    d <- replicate(nrep, { Xp <- X; Xp[, j] <- X[sample(nrow(X)), j]; base - r2(yy, predfun(Xp)) })
    c(mean = mean(d), sd = sd(d)) })) }
grouped_perm <- function(predfun, X, yy, groups, nrep = 20, seed = 0) {
  set.seed(seed); base <- r2(yy, predfun(X))
  t(sapply(groups, function(cols) {
    d <- replicate(nrep, { Xp <- X; p <- sample(nrow(X)); Xp[, cols] <- X[p, cols]
                           base - r2(yy, predfun(Xp)) })
    c(mean = mean(d), sd = sd(d)) })) }
PI <- perm_imp(gp, Xd[ite, ], y[ite]); rownames(PI) <- DESC
knitr::kable(round(PI[order(-PI[, "mean"]), ], 4),
             caption = "Permutation importance (drop in test R2, 20 repeats)")
G <- list("MolWt alone" = 1, "HeavyAtoms alone" = 9, "MolWt + HeavyAtoms" = c(1, 9),
          "size group (MolWt, HeavyAtoms, RingCount, RotB)" = c(1, 9, 8, 6))
cat(sprintf("correlation MolWt vs HeavyAtoms = %.3f\n", cor(Xd[, 1], Xd[, 9])))
GP <- grouped_perm(gp, Xd[ite, ], y[ite], G)
knitr::kable(round(GP, 4), caption = "Grouped permutation: correlated features mask each other")


In [ ]:
d1 <- data.frame(feature = factor(DESC, levels = DESC[order(PI[, "mean"])]),
                 impurity = rfn$variable.importance[1:11]/sum(rfn$variable.importance),
                 permutation = PI[, "mean"], sd = PI[, "sd"])
g1 <- ggplot(d1) +
  geom_col(aes(feature, impurity, fill = "impurity (biased, with noise columns)"),
           width = .38, position = position_nudge(x = -.2)) +
  geom_col(aes(feature, permutation, fill = "permutation (test set)"),
           width = .38, position = position_nudge(x = .2)) +
  geom_errorbar(aes(x = feature, ymin = permutation - sd, ymax = permutation + sd),
                width = .15, position = position_nudge(x = .2), linewidth = .3) +
  coord_flip() + scale_fill_manual(values = unname(PAL[c("orange", "blue")])) +
  labs(x = NULL, y = "importance (different units)", title = "Two importances, two stories") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
d2 <- data.frame(what = factor(rownames(GP), levels = rownames(GP)), m = GP[, "mean"], s = GP[, "sd"])
g2 <- ggplot(d2, aes(what, m)) +
  geom_col(fill = c("grey65", "grey65", PAL["blue"], PAL["green"])) +
  geom_errorbar(aes(ymin = m - s, ymax = m + s), width = .2, linewidth = .3) + coord_flip() +
  labs(x = NULL, y = "drop in test R^2 when permuted", title = "Correlated features mask each other") +
  theme(axis.text.y = element_text(size = 6.5))
print(g1 | g2)



<p class="caption"><b>Figure 1.</b> Left: impurity importance (orange, from the forest trained with two noise columns, rescaled to share of total) against permutation importance measured on the test set (blue, with its standard deviation over 20 repeats). The rankings differ substantially — TPSA and MolWt are near the top by impurity and mid-table by permutation. Right: permuting molecular weight alone costs 0.072 R² and heavy-atom count alone 0.025, the two together 0.084 — but the whole four-descriptor "size" group costs 0.223, more than twice the sum of its parts.</p>

The grouped numbers are the honest answer to "how much does size matter to this model": not the 0.02–0.07 that either column shows alone, but the 0.22 the group shows together. When you report permutation importances for correlated descriptors, report the groups; when you report single-feature values, say that correlated features share credit.

<div class="exercise"><h4>Exercise 1</h4>
<p>Compute permutation importance of the <em>same</em> gradient-boosting model on the <strong>training</strong> set instead of the test set (<code>perm_imp(gp, Xd[itr, ], y[itr])</code>) and compare the ranking with the test-set one by Spearman correlation. Which features gain importance, and what does the difference tell you about what the model memorised? (Hint: a feature that matters only on training data is fitting noise.)</p></div>

## 5. Partial dependence, ICE and ALE

Importance says *how much*; dependence says *in which direction and with what shape*. The **partial dependence** of the prediction on feature \(j\)<sup class="c"><a href="#r6">[6]</a></sup> is

\[ \widehat{\mathrm{PD}}_j(v) = \frac{1}{n}\sum_{i=1}^{n} \hat f\big(v, \mathbf{x}^{(i)}_{-j}\big) ,\]

i.e. set feature \(j\) to \(v\) for *every* row, keep the other features as they are, and average the predictions. The flaw is visible in the formula: it evaluates the model at combinations that may not exist. Forcing `MolWt = 200` on a molecule with 40 heavy atoms asks about an impossible compound.

**ICE** curves<sup class="c"><a href="#r7">[7]</a></sup> keep the individual lines instead of averaging them, exposing heterogeneity: a flat PDP can hide half the molecules rising and half falling. **ALE**<sup class="c"><a href="#r8">[8]</a></sup> fixes the extrapolation problem differently: partition the feature's range into intervals and, inside each, compute the average *difference* in prediction when the feature is moved from the interval's lower to its upper edge, using only the rows that fall in that interval. Accumulating those local differences gives a curve that never leaves the data. `iml::FeatureEffect` computes all three; ALE is also written out by hand below, in eight lines, because that is the whole method.



In [ ]:
dte <- as.data.frame(Xd[ite, ]); names(dte) <- DESC
pred_gbm <- Predictor$new(predict.function = function(model, newdata) gp(as.matrix(newdata)),
                          model = gbm, data = dte, y = y[ite])
JF <- "FractionCSP3"
fe_pdp <- FeatureEffect$new(pred_gbm, feature = JF, method = "pdp+ice", grid.size = 25)
fe_ale <- FeatureEffect$new(pred_gbm, feature = JF, method = "ale", grid.size = 20)
ale_1d <- function(predfun, X, j, n_bins = 20) {        # first-order accumulated local effects
  edges <- unique(quantile(X[, j], seq(0, 1, length.out = n_bins + 1)))
  wb <- pmin(pmax(findInterval(X[, j], edges, left.open = TRUE), 1), length(edges) - 1)
  local <- numeric(length(edges) - 1)
  for (k in seq_len(length(edges) - 1)) {
    m <- wb == k; if (!any(m)) next
    lo <- X[m, , drop = FALSE]; hi <- lo; lo[, j] <- edges[k]; hi[, j] <- edges[k + 1]
    local[k] <- mean(predfun(hi) - predfun(lo)) }
  curve <- c(0, cumsum(local)); counts <- tabulate(wb, length(edges) - 1)
  curve <- curve - sum((head(curve, -1) + tail(curve, -1))/2*counts)/sum(counts)   # centre on the data
  data.frame(x = edges, ale = curve) }
a1 <- ale_1d(gp, Xd[ite, ], which(DESC == JF))
pdp <- subset(fe_pdp$results, .type == "pdp"); ice <- subset(fe_pdp$results, .type == "ice")
cat(sprintf("feature: %s\nPDP spans %.3f log units (%.2f to %.2f)\n", JF,
            diff(range(pdp$.value)), min(pdp$.value), max(pdp$.value)))
cat(sprintf("iml ALE spans %.3f log units; the hand-written ALE spans %.3f\n",
            diff(range(fe_ale$results$.value)), diff(range(a1$ale))))
rise <- mean(tapply(ice$.value, ice$.id, function(v) tail(v, 1) > head(v, 1)))
cat(sprintf("ICE: %.0f%% of test molecules rise from the lowest to the highest %s, %.0f%% fall\n",
            100*rise, JF, 100*(1 - rise)))


In [ ]:
icec <- do.call(rbind, lapply(split(ice, ice$.id), function(d) { d$.value <- d$.value - mean(d$.value); d }))
g1 <- ggplot() +
  geom_line(data = subset(icec, as.integer(.id) %% 12 == 0),
            aes(.data[[JF]], .value, group = .id), colour = "grey80", linewidth = .2) +
  geom_line(data = transform(pdp, c = .value - mean(.value)), aes(.data[[JF]], c),
            colour = PAL["orange"], linewidth = 1) +
  labs(x = JF, y = "centred prediction", title = "ICE curves and their average (PDP)")
g2 <- ggplot() +
  geom_line(data = transform(pdp, c = .value - mean(.value)), aes(.data[[JF]], c, colour = "PDP")) +
  geom_line(data = a1, aes(x, ale, colour = "ALE (hand-written)")) +
  geom_line(data = fe_ale$results, aes(.data[[JF]], .value, colour = "ALE (iml)"), linetype = 2) +
  scale_colour_manual(values = unname(PAL[c("green", "blue", "orange")])) +
  labs(x = JF, y = "effect (log units)", title = "PDP vs ALE, same model") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
pw <- FeatureEffect$new(pred_gbm, feature = "MolWt", method = "pdp", grid.size = 25)
aw <- ale_1d(gp, Xd[ite, ], 1)
g3 <- ggplot() +
  geom_line(data = transform(pw$results, c = .value - mean(.value)), aes(MolWt, c, colour = "PDP")) +
  geom_line(data = aw, aes(x, ale, colour = "ALE")) +
  scale_colour_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = "MolWt", y = "effect (log units)", title = "MolWt: correlated with HeavyAtoms (r = 0.98)") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
print(g1 | g2 | g3)
cat(sprintf("MolWt: PDP spans %.3f, hand-written ALE spans %.3f log units\n",
            diff(range(pw$results$.value)), diff(range(aw$ale))))



<p class="caption"><b>Figure 2.</b> Left: individual conditional expectation curves (grey, every twelfth test molecule, each centred on its own mean) with the partial-dependence curve in orange; the PDP is the average of lines that do not all move the same way — 73 % rise, 27 % fall. Centre: PDP against the hand-written ALE and <code>iml</code>'s ALE for the same feature; the two ALE implementations agree to within 0.003 log units and both span slightly less than the PDP. Right: the same comparison for molecular weight, whose correlation with heavy-atom count at r = 0.98 makes the PDP's "hold everything else fixed" assumption indefensible; here ALE spans 0.651 log units against the PDP's 0.458.</p>

Read the three panels as a procedure: compute ICE to see whether an average is meaningful at all, compute ALE whenever the feature is correlated with others, and reserve the PDP for features close to independent of the rest. The molecular-weight panel is the instructive one — the two curves disagree by 40 % in span on a feature that *is* strongly correlated, exactly where the theory predicts they must. Molnar and colleagues' catalogue of pitfalls<sup class="c"><a href="#r26">[26]</a></sup> is the compact reference for which curve to trust when.

## 6. Shapley values in one page

Shapley's problem<sup class="c"><a href="#r9">[9]</a></sup> was how to divide the payoff of a cooperative game fairly among its players. Lundberg and Lee's insight<sup class="c"><a href="#r10">[10]</a></sup> was that a prediction is such a payoff and the features are the players: the "payoff" of a coalition \(S\) of features is

\[ v(S) = \mathbb{E}\big[\hat f(\mathbf{X}) \mid \mathbf{X}_S = \mathbf{x}_S\big] - \mathbb{E}\big[\hat f(\mathbf{X})\big], \]

estimated in practice by replacing the features *not* in \(S\) with values drawn from a background sample. The Shapley value of feature \(j\) is its average marginal contribution over all orderings in which the coalition could be assembled:

\[ \phi_j = \sum_{S \subseteq F \setminus \{j\}} \frac{|S|!\,(|F|-|S|-1)!}{|F|!}\,\big[v(S \cup \{j\}) - v(S)\big]. \]

It is the *unique* attribution satisfying four axioms — **efficiency** (\(\sum_j \phi_j = \hat f(\mathbf{x}) - \mathbb{E}[\hat f]\), the explanation adds up to the prediction), **symmetry**, **dummy** and **additivity**. The price is the sum over all \(2^{|F|-1}\) coalitions.

<div class="box bridge"><h4>Substitution — no TreeSHAP in this environment</h4>
<p>The Python edition uses <strong>TreeSHAP</strong><sup class="c"><a href="#r11">[11]</a></sup>, which computes exact Shapley values for a tree ensemble in polynomial time. No R port of TreeSHAP is installed here, so this unit uses two alternatives and says which is which. <code>iml::Shapley</code><sup class="c"><a href="#r29">[29]</a></sup> estimates the <em>same</em> quantity by <strong>Monte-Carlo sampling over feature orderings</strong>: it is unbiased but noisy, and its error is visible in the efficiency check below (the attributions sum to the prediction only up to sampling error, where TreeSHAP's sum is exact to machine precision). <code>DALEX::predict_parts</code><sup class="c"><a href="#r32">[32]</a></sup> with <code>type = "break_down"</code> computes a different decomposition — a single greedy ordering of features rather than an average over all orderings — which is cheaper, always sums exactly, and is <em>not</em> a Shapley value. Section 6.1 anchors both against an exact enumeration on a four-feature model, which is the only fully trustworthy number in the section.</p></div>

### 6.1 Worked example: exact Shapley values by enumeration

With four features there are only \(2^3 = 8\) coalitions per feature, so the sum can be evaluated in full. We fit a small model on four descriptors, choose one test compound, and compute all four \(\phi_j\) by brute force with a 100-molecule background sample — then ask `iml::Shapley` for the same numbers. The axioms predict that the exact values must add up to \(\hat f(\mathbf{x}) - \mathbb{E}[\hat f]\).



In [ ]:
F4 <- match(c("MolLogP", "TPSA", "RotB", "QED"), DESC)
set.seed(0)
g4 <- xgb.train(list(eta = 0.05, max_depth = 4, objective = "reg:squarederror", nthread = 4),
                xgb.DMatrix(Xd[itr, F4], label = y[itr]), nrounds = 200, verbose = 0)
g4p <- function(X) predict(g4, xgb.DMatrix(as.matrix(X)))
set.seed(0); bg <- Xd[itr, ][sample(length(itr), 100), F4]; colnames(bg) <- DESC[F4]
xq <- Xd[ite, ][1, F4]
vS <- function(S) { Z <- bg; if (length(S)) for (j in S) Z[, j] <- xq[j]
                    mean(g4p(Z)) - mean(g4p(bg)) }
exact_shapley <- function(j, n = 4) {
  others <- setdiff(1:n, j); tot <- 0
  for (r in 0:(n - 1)) {
    combs <- if (r == 0) list(integer(0)) else combn(others, r, simplify = FALSE)
    for (S in combs) tot <- tot +
      factorial(length(S))*factorial(n - length(S) - 1)/factorial(n)*(vS(c(S, j)) - vS(S)) }
  tot }
phi_exact <- sapply(1:4, exact_shapley)
pred4 <- Predictor$new(predict.function = function(model, newdata) g4p(as.matrix(newdata)),
                       model = g4, data = as.data.frame(bg))
set.seed(0)
sh <- Shapley$new(pred4, x.interest = as.data.frame(matrix(xq, 1, dimnames = list(NULL, DESC[F4]))),
                  sample.size = 200)
phi_mc <- sh$results$phi[match(DESC[F4], sh$results$feature)]
fx <- g4p(matrix(xq, 1)); Ef <- mean(g4p(bg))
knitr::kable(data.frame(feature = DESC[F4], value = xq, exact_enumeration = phi_exact,
                        iml_Shapley_MC = phi_mc), digits = 4, row.names = FALSE,
             caption = "Shapley values of one compound: exact and Monte-Carlo")
cat(sprintf("efficiency axiom:  sum(phi_exact) = %+.4f   f(x) - E[f] = %+.4f   difference %.1e\n",
            sum(phi_exact), fx - Ef, abs(sum(phi_exact) - (fx - Ef))))
cat(sprintf("Monte-Carlo (200 samples): max |exact - MC| = %.4f, correlation %.3f, sum %+.4f\n",
            max(abs(phi_exact - phi_mc)), cor(phi_exact, phi_mc), sum(phi_mc)))



The exact enumeration satisfies efficiency to machine precision — the printed difference is \(0\) — and that is the verification worth doing once in your life: after it, a Shapley value is not a black box inside a black box but a quantity you have computed by hand. The Monte-Carlo estimate from `iml` reproduces the ordering perfectly (correlation 0.997) but individual values differ by up to about 0.11 log units with 200 samples. **Report the sample size, and check the efficiency gap**: it is the honest error bar on a sampled Shapley value, and it is absent from most published figures.

Two further cautions travel with Shapley values. The value function above is **interventional** — features outside the coalition are replaced by independent background draws — which, exactly as with the PDP, evaluates the model off the data manifold when features are correlated; the *conditional* alternative requires modelling the feature distribution and gives different, sometimes contradictory, attributions<sup class="c"><a href="#r12">[12]</a></sup>. And a Shapley value is an attribution of the *prediction*, never of the outcome. Rodriguez-Perez and Bajorath's study of compound-activity predictions is the chemistry-specific reading of both cautions<sup class="c"><a href="#r15">[15]</a></sup>.

### 6.2 Global and local readings on the real model



In [ ]:
set.seed(0); shsub <- sample(nrow(dte), 120)       # Monte-Carlo Shapley is expensive: a subsample
t0 <- proc.time()[["elapsed"]]
PHI <- t(sapply(shsub, function(i) { set.seed(i)
  s <- Shapley$new(pred_gbm, x.interest = dte[i, , drop = FALSE], sample.size = 60)
  s$results$phi[match(DESC, s$results$feature)] }))
colnames(PHI) <- DESC
cat(sprintf("Monte-Carlo Shapley for %d test compounds in %.0f s\n", length(shsub),
            proc.time()[["elapsed"]] - t0))
glob <- data.frame(descriptor = DESC, mean_abs_phi = colMeans(abs(PHI)), mean_phi = colMeans(PHI),
                   permutation_importance = PI[, "mean"])
knitr::kable(glob[order(-glob$mean_abs_phi), ], digits = 4, row.names = FALSE,
             caption = "Global Shapley summary against permutation importance")
cat(sprintf("Spearman between the |Shapley| ranking and the permutation ranking: %.3f\n",
            cor(colMeans(abs(PHI)), PI[, "mean"], method = "spearman")))
cat(sprintf("efficiency gap at 60 samples: mean |sum(phi) - (f(x) - E[f])| = %.4f log units\n",
            mean(abs(rowSums(PHI) - (gp(Xd[ite, ][shsub, ]) - mean(gp(Xd[ite, ])))))))


In [ ]:
long <- do.call(rbind, lapply(DESC, function(d) data.frame(
  feature = d, phi = PHI[, d],
  value = (Xd[ite, ][shsub, d] - min(Xd[ite, d]))/diff(range(Xd[ite, d])))))
long$feature <- factor(long$feature, levels = DESC[order(colMeans(abs(PHI)))])
g1 <- ggplot(long, aes(phi, feature, colour = value)) +
  geom_point(position = position_jitter(height = .22), size = .7, alpha = .8) +
  geom_vline(xintercept = 0, linewidth = .3) + scale_colour_viridis_c(name = "feature value\n(scaled)") +
  labs(x = "Shapley value (log units)", y = NULL,
       title = "Shapley summary: 120 test compounds, 11 descriptors") +
  theme(legend.position = "right", legend.key.height = unit(.4, "cm"), legend.title = element_text(size = 6))
print(g1)



<p class="caption"><b>Figure 3.</b> Shapley summary plot: one dot per sampled test compound per descriptor, horizontal position the contribution to that compound's prediction in log units, colour the descriptor's own value. Features are ordered by mean absolute contribution, with the number of hydrogen-bond acceptors (0.179 log units) first and the count of donors (0.057) last. A clean colour gradient from left to right would mean a monotone effect; most rows here are colour-mixed, meaning the effect depends on the other features.</p>



In [ ]:
i0 <- which.max(gp(Xd[ite, ]))
set.seed(1); sh0 <- Shapley$new(pred_gbm, x.interest = dte[i0, , drop = FALSE], sample.size = 400)
phi0 <- sh0$results$phi[match(DESC, sh0$results$feature)]
expl <- DALEX::explain(gbm, data = dte, y = y[ite], verbose = FALSE, label = "xgboost",
                       predict_function = function(m, d) gp(as.matrix(d)))
bd <- DALEX::predict_parts(expl, new_observation = dte[i0, , drop = FALSE], type = "break_down")
bdv <- bd$contribution[bd$variable_name %in% DESC]; names(bdv) <- bd$variable_name[bd$variable_name %in% DESC]
cat(sprintf("compound explained: test row %d, measured pActivity %.2f, predicted %.2f, base value %.2f\n",
            i0, y[ite][i0], gp(Xd[ite, ][i0, , drop = FALSE]), mean(gp(Xd[ite, ]))))
cat("three largest Shapley contributions: ",
    paste(sprintf("%s %+.3f", DESC[order(-abs(phi0))][1:3], phi0[order(-abs(phi0))][1:3]),
          collapse = ", "), "\n")
cat(sprintf("DALEX break-down sums to %+.3f (exactly the prediction minus the intercept)\n", sum(bdv)))
cat(sprintf("Spearman between iml Shapley and DALEX break-down contributions: %.3f\n",
            cor(phi0, bdv[DESC], method = "spearman")))


In [ ]:
dep <- data.frame(x = Xd[ite, ][shsub, "HBA"], phi = PHI[, "HBA"], RotB = Xd[ite, ][shsub, "RotB"])
g1 <- ggplot(dep, aes(x, phi, colour = RotB)) + geom_point(size = 1) +
  geom_hline(yintercept = 0, linewidth = .3) + scale_colour_viridis_c() +
  labs(x = "HBA", y = "Shapley value for HBA (log units)", title = "Dependence plot: HBA") +
  theme(legend.position = "right", legend.key.height = unit(.4, "cm"))
wf <- data.frame(feature = DESC, phi = phi0, bd = bdv[DESC])
wf <- wf[order(-abs(wf$phi)), ][1:8, ]
wf$feature <- factor(wf$feature, levels = rev(wf$feature))
g2 <- ggplot(wf) +
  geom_col(aes(feature, phi, fill = "iml Shapley (MC, 400)"), width = .4,
           position = position_nudge(x = -.2)) +
  geom_col(aes(feature, bd, fill = "DALEX break-down"), width = .4, position = position_nudge(x = .2)) +
  coord_flip() + geom_hline(yintercept = 0, linewidth = .3) +
  scale_fill_manual(values = unname(PAL[c("orange", "blue")])) +
  labs(x = NULL, y = "contribution to the prediction (log units)",
       title = sprintf("One compound: predicted %.2f, measured %.2f", gp(Xd[ite, ][i0, , drop = FALSE]),
                       y[ite][i0])) +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
print(g1 | g2)



<p class="caption"><b>Figure 4.</b> Left: Shapley dependence plot for the most important descriptor, coloured by the number of rotatable bonds; the vertical spread at a fixed x value is interaction with the other descriptors. Right: the two local decompositions of the most potent predicted test compound (measured 9.10, predicted 8.47, base value 6.50), for its eight largest contributions. They agree on the sign of most features and disagree on magnitudes — their rank correlation is 0.76 — because a break-down uses one greedy ordering while a Shapley value averages over all of them.</p>

The global Shapley ranking and the permutation ranking agree closely (Spearman 0.873), which is reassuring but not independent evidence — both are driven by the same model. Agreement between two explanation methods means the explanation is a stable property of the model, not that it is a true property of chemistry. Note also the efficiency gap printed above: with 60 Monte-Carlo samples per compound the attributions miss the prediction by about 0.12 log units on average, which is the scale at which a difference between two features is noise.

## 7. LIME: a local surrogate, written by hand

LIME<sup class="c"><a href="#r13">[13]</a></sup> answers "why this prediction" differently: perturb the instance, ask the black box for predictions on the perturbed points, weight them by proximity to the original, and fit a *sparse linear model* to those weighted predictions. The coefficients of that local model are the explanation. In a dozen lines:



In [ ]:
lime_explain <- function(predfun, x, X_ref, n = 2000, sigma = 0.5, kw = 0.75, seed = 0) {
  set.seed(seed); sdv <- apply(X_ref, 2, sd)
  Z <- matrix(rep(x, each = n), n) + matrix(rnorm(n*length(x), 0, 1), n) %*% diag(sigma*sdv)
  f <- predfun(Z)                                              # the black box's answers
  d <- sqrt(rowSums(sweep(sweep(Z, 2, x), 2, sdv, "/")^2)/length(x))   # normalised distance
  w <- exp(-(d^2)/kw^2)                                        # proximity weights
  Zs <- sweep(sweep(Z, 2, colMeans(X_ref)), 2, sdv, "/")
  fit <- glmnet(Zs, f, alpha = 0, lambda = 0.01, weights = w)  # weighted ridge surrogate
  p <- as.numeric(predict(fit, Zs))
  list(coef = as.numeric(coef(fit))[-1],
       r2 = 1 - sum(w*(f - p)^2)/sum(w*(f - weighted.mean(f, w))^2)) }
L <- lapply(0:2, function(s) lime_explain(gp, Xd[ite, ][i0, ], Xd[itr, ], seed = s))
cat(sprintf("local surrogate R2 (how well the linear model imitates the black box nearby): %.3f, %.3f, %.3f\n",
            L[[1]]$r2, L[[2]]$r2, L[[3]]$r2))
for (k in 1:3) cat(sprintf("top three features, seed %d: %s\n", k - 1,
                           paste(DESC[order(-abs(L[[k]]$coef))][1:3], collapse = ", ")))
cat(sprintf("Spearman between seeds 0 and 1: %.3f; 0 and 2: %.3f\n",
            cor(L[[1]]$coef, L[[2]]$coef, method = "spearman"),
            cor(L[[1]]$coef, L[[3]]$coef, method = "spearman")))
cat(sprintf("Spearman between LIME (seed 0) and this compound's Shapley values: %.3f\n",
            cor(L[[1]]$coef, phi0, method = "spearman")))


In [ ]:
ld <- do.call(rbind, lapply(1:3, function(k)
  data.frame(feature = DESC, coef = L[[k]]$coef, seed = paste("seed", k - 1))))
ld$feature <- factor(ld$feature, levels = DESC)
g1 <- ggplot(ld, aes(feature, coef, fill = seed)) +
  geom_col(position = position_dodge(width = .8), width = .75) + coord_flip() +
  scale_fill_manual(values = unname(PAL[c("blue", "orange", "green")])) +
  geom_hline(yintercept = 0, linewidth = .3) +
  labs(x = NULL, y = "LIME coefficient", title = "LIME depends on its random sample") +
  theme(legend.position = "bottom", legend.title = element_blank(), legend.text = element_text(size = 6.5))
sd0 <- data.frame(feature = factor(DESC, levels = DESC[order(abs(phi0))]), phi = phi0)
g2 <- ggplot(sd0, aes(feature, phi, fill = phi > 0)) + geom_col(show.legend = FALSE) + coord_flip() +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
  geom_hline(yintercept = 0, linewidth = .3) +
  labs(x = NULL, y = "Shapley value (log units)", title = "Shapley values for the same compound")
print(g1 | g2)



<p class="caption"><b>Figure 5.</b> Left: LIME coefficients for one compound from three different random perturbation samples; with 2,000 perturbations the three runs are close (rank correlations 0.98 and 0.99) and only the third-ranked feature changes between them. Right: the Monte-Carlo Shapley values of the same compound. The two methods rank the descriptors almost independently — Spearman 0.11 — even though both claim to explain the same prediction.</p>

Three practical rules follow. Always report the local fit quality: the surrogate here reproduces only about 40 % of the black box's variance in the neighbourhood, so it is an approximation of an approximation, and a LIME explanation quoted without its \(R^2\) hides that. Always re-run with several seeds and show the variability — stability improves with the number of perturbations, so a stable LIME is partly a statement about your compute budget. And note the disagreement with Shapley: two defensible local explanations of the *same prediction of the same model* put different features first. Remember also that both LIME and sampling-based Shapley build their perturbations by sampling features independently, which makes them manipulable: an adversary can construct a model that behaves one way on real data and another on the off-manifold points the explainer samples<sup class="c"><a href="#r14">[14]</a></sup>.

## 8. Counterfactual explanations

A counterfactual answers the question a chemist actually asks: *what would have to change for this compound to be predicted potent?* Formally, find \(\mathbf{x}'\) close to \(\mathbf{x}\) with \(\hat f(\mathbf{x}') \ge \tau\), preferring few changed features and values inside the data range<sup class="c"><a href="#r17">[17]</a></sup>. Below we search single-descriptor changes for a compound the logistic classifier calls inactive.



In [ ]:
pprob <- function(Z) as.numeric(predict(lg, newx = Z, s = "lambda.min", type = "response"))
ic <- which(pprob(Zd[ite, ]) < 0.35)[1]
x0 <- Xd[ite, ][ic, ]; p0 <- pprob(matrix((x0 - mu)/sg, 1))
lo <- apply(Xd[itr, ], 2, min); hi <- apply(Xd[itr, ], 2, max); sdtr <- apply(Xd[itr, ], 2, sd)
rows <- list()
for (j in 1:11) {
  grid <- seq(lo[j], hi[j], length.out = 60)
  Xtry <- matrix(rep(x0, each = 60), 60); Xtry[, j] <- grid
  p <- pprob(sweep(sweep(Xtry, 2, mu), 2, sg, "/")); ok <- which(p >= 0.5)
  if (length(ok)) { k <- ok[which.min(abs(grid[ok] - x0[j]))]
    rows[[length(rows) + 1]] <- data.frame(descriptor = DESC[j], from = x0[j], to = grid[k],
      change_in_SD = abs(grid[k] - x0[j])/sdtr[j], new_P_potent = p[k]) } }
cf <- do.call(rbind, rows); cf <- cf[order(cf$change_in_SD), ]
cat(sprintf("compound: test row %d, measured pActivity %.2f, P(potent) = %.3f\n", ic, y[ite][ic], p0))
knitr::kable(cf, digits = 3, row.names = FALSE,
             caption = "Single-descriptor counterfactuals that reach P(potent) >= 0.5")
cat(sprintf("the cheapest single change is %s (%.2f -> %.2f, %.2f standard deviations)\n",
            cf$descriptor[1], cf$from[1], cf$to[1], cf$change_in_SD[1]))



The table is useful and dangerous in equal measure. Useful, because "add three rotatable bonds" is a sentence a chemist can act on. Dangerous, because the counterfactual lives in *descriptor space*, and nothing guarantees that a molecule with those descriptor values exists, let alone that it can be synthesised or keeps the rest of the molecule's properties. Read the last rows of the table for the reductio: the model will also call this compound potent if its molecular weight rises to 823 Da or it acquires eleven rings — formally valid counterfactuals, chemically absurd. A counterfactual in descriptor space is a hypothesis about the model; a counterfactual in chemistry is a compound someone has to make. State which one you are reporting.

## 9. Mapping explanations back to the molecule

For a fingerprint model the features are hashed bits, and an explanation in bit numbers is useless to a chemist. The repair in the Python edition is mechanical: a Morgan bit records a circular atom environment<sup class="c"><a href="#r18">[18]</a></sup>, and RDKit reports, for each set bit, the central atom and the radius that produced it, so each bit's attribution can be distributed over the atoms of its environment and drawn on the molecule — the idea behind Riniker and Landrum's similarity maps<sup class="c"><a href="#r21">[21]</a></sup>.

<div class="box bridge"><h4>Substitution — what R can and cannot do here</h4>
<p><strong>The bit-to-atom mapping cannot be recomputed in this environment.</strong> It requires the fingerprint generator's bit-information map, which only RDKit (or an equivalent toolkit) produces; the frozen file <code>data/rock1_morgan_sparse.csv.gz</code> stores <em>which</em> bits are set, not which atoms set them, and <code>data/rock1_curated.sdf.gz</code> stores the structures but not the hash. So no molecule is coloured in this unit. What the frozen data <em>does</em> allow is the layer underneath the picture, and it is worth doing explicitly: identify the bits the model relies on, count how many compounds carry each, and measure how potency is distributed among the carriers. That is a testable statement about the model and the library; the step from a bit to a named substructure needs the toolkit, and the SDF is shipped precisely so that a reader with RDKit, ChemmineR or rcdk can take it.</p></div>



In [ ]:
set.seed(0)
rf_imp <- ranger(x = Xfp[itr, ], y = factor(ycls[itr]), num.trees = 300, min.node.size = 2,
                 probability = TRUE, importance = "impurity", seed = 1, num.threads = 4)
bi <- sort(rf_imp$variable.importance, decreasing = TRUE)[1:10]
bits <- as.integer(sub("b", "", names(bi)))
tab <- data.frame(bit = bits, impurity_importance = as.numeric(bi),
                  compounds_with_bit = sapply(bits, function(b) sum(Xfp[, paste0("b", b)])),
                  pct_of_library = 100*sapply(bits, function(b) mean(Xfp[, paste0("b", b)])),
                  pct_potent_when_set = 100*sapply(bits, function(b) mean(ycls[Xfp[, paste0("b", b)] == 1])))
knitr::kable(tab, digits = 1, row.names = FALSE,
             caption = "The ten bits the fingerprint forest relies on most")
cat(sprintf("library-wide potent fraction: %.1f%%\n", 100*mean(ycls)))
cat(sprintf("mean bits set per molecule %.1f of 2048; the ten bits above are carried by %d distinct compounds (%.0f%% of the library)\n",
            mean(rowSums(Xfp)), sum(rowSums(Xfp[, paste0("b", bits)]) > 0),
            100*mean(rowSums(Xfp[, paste0("b", bits)]) > 0)))


In [ ]:
g1 <- ggplot(tab, aes(pct_of_library, pct_potent_when_set, size = impurity_importance)) +
  geom_point(colour = PAL["blue"], alpha = .8) +
  ggrepel::geom_text_repel(aes(label = bit), size = 2.4, max.overlaps = 20) +
  geom_hline(yintercept = 100*mean(ycls), linetype = 2, colour = PAL["orange"]) +
  labs(x = "% of the library carrying the bit", y = "% potent among carriers",
       size = "impurity", title = "Ten most-used bits: prevalence vs enrichment") +
  theme(legend.position = "right", legend.key.height = unit(.3, "cm"), legend.title = element_text(size = 6))
bb <- bits[1]
dd <- data.frame(pAct = y, carrier = factor(Xfp[, paste0("b", bb)], 0:1, c("bit absent", "bit present")))
g2 <- ggplot(dd, aes(carrier, pAct, fill = carrier)) +
  geom_violin(alpha = .6, show.legend = FALSE) + geom_boxplot(width = .15, outlier.size = .4, show.legend = FALSE) +
  scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
  labs(x = NULL, y = "pActivity", title = sprintf("Potency distribution for bit %d", bb))
print(g1 | g2)
cat(sprintf("bit %d: median pActivity %.2f when present vs %.2f when absent (Wilcoxon p = %.1e)\n", bb,
            median(y[Xfp[, paste0("b", bb)] == 1]), median(y[Xfp[, paste0("b", bb)] == 0]),
            wilcox.test(y ~ Xfp[, paste0("b", bb)])$p.value))



<p class="caption"><b>Figure 6.</b> Left: the ten bits with the largest impurity importance in the fingerprint forest, plotted as prevalence against the fraction of their carriers that are potent, with the library-wide potent fraction (33.1 %) as the dashed line; all ten lie above it, from bit 1603 at 39.9 % to bit 1365 at 61.9 %, and the two most enriched are carried by only 13 % and 18 % of the library. Right: the full potency distribution of compounds with and without the single most important bit, 671.</p>

Now the check that matters: **does this agree with the known SAR?** It cannot, by itself. The table says that the model leans on a handful of bits carried by 11–52 % of the library whose carriers are up to 1.9 times as likely to be potent as the library average — a correct, verifiable statement about the model. It does *not* say that those bits encode the hinge-binding aminopyridine motif of ROCK inhibitors<sup class="c"><a href="#r22">[22]</a></sup>, because the hash cannot be inverted here, and even with RDKit the step from "the model used this substructure" to "this substructure binds the hinge" needs a control: the same nitrogen-bearing heterocycles are the most frequent substructures in the training set, so a model that had learned nothing but their frequency would highlight the same bits. Polishchuk's review of QSAR interpretation<sup class="c"><a href="#r27">[27]</a></sup> makes the general version of this point: a structural interpretation is only as strong as the control you run against it, which is the subject of the next section.

<div class="exercise"><h4>Exercise 2</h4>
<p>Take the three most enriched bits from the table above and compute their pairwise co-occurrence (<code>crossprod(Xfp[, c("b1365","b1415","b671")])</code>). If two bits are set in almost the same compounds, the forest's reliance on both is one piece of evidence, not two. Then recompute the enrichment restricted to the largest Butina cluster of Unit 15: does the bit still separate potent from non-potent compounds <em>within</em> one series, or only between series?</p></div>

## 10. The sanity checks

An explanation method always returns an explanation. The only way to know whether yours carries information is to run controls, exactly as for a model's performance.

**Control 1: explain a model that cannot have learned anything.** Permute the labels (y-scrambling<sup class="c"><a href="#r23">[23]</a></sup>), refit the identical pipeline, and explain it. The scrambled model has no predictive power on held-out data by construction; if its explanation still looks chemically reasonable, then "looks reasonable" is not evidence.



In [ ]:
set.seed(0); y_scram <- sample(y[itr])
set.seed(0); gbm_s <- xgb.train(PAR, xgb.DMatrix(Xd[itr, ], label = y_scram), nrounds = 400, verbose = 0)
gps <- function(X) predict(gbm_s, xgb.DMatrix(as.matrix(X)))
cat(sprintf("real model      : train R2 %.3f  test R2 %.3f\n",
            r2(y[itr], gp(Xd[itr, ])), r2(y[ite], gp(Xd[ite, ]))))
cat(sprintf("scrambled model : train R2 %.3f  test R2 %.3f  <- no predictive power\n",
            r2(y_scram, gps(Xd[itr, ])), r2(y[ite], gps(Xd[ite, ]))))
pred_s <- Predictor$new(predict.function = function(model, newdata) gps(as.matrix(newdata)),
                        model = gbm_s, data = dte, y = y[ite])
PHI_s <- t(sapply(shsub, function(i) { set.seed(i)
  s <- Shapley$new(pred_s, x.interest = dte[i, , drop = FALSE], sample.size = 60)
  s$results$phi[match(DESC, s$results$feature)] }))
colnames(PHI_s) <- DESC
cmpd <- data.frame(descriptor = DESC, mean_abs_real = colMeans(abs(PHI)),
                   mean_abs_scrambled = colMeans(abs(PHI_s)))
cmpd$rank_real <- rank(-cmpd$mean_abs_real); cmpd$rank_scrambled <- rank(-cmpd$mean_abs_scrambled)
knitr::kable(cmpd, digits = 3, row.names = FALSE, caption = "Attribution of a real and a scrambled model")
cat(sprintf("total attribution mass: real %.3f, scrambled %.3f log units (%.0f%% of the real model's)\n",
            sum(colMeans(abs(PHI))), sum(colMeans(abs(PHI_s))),
            100*sum(colMeans(abs(PHI_s)))/sum(colMeans(abs(PHI)))))
cat(sprintf("Spearman between the two rankings: %.3f; the scrambled model's top three are %s\n",
            cor(colMeans(abs(PHI)), colMeans(abs(PHI_s)), method = "spearman"),
            paste(DESC[order(-colMeans(abs(PHI_s)))][1:3], collapse = ", ")))


In [ ]:
sc <- rbind(data.frame(feature = DESC, m = colMeans(abs(PHI)),
                       model = sprintf("real (test R^2 = %.2f)", r2(y[ite], gp(Xd[ite, ])))),
            data.frame(feature = DESC, m = colMeans(abs(PHI_s)),
                       model = sprintf("y-scrambled (test R^2 = %.2f)", r2(y[ite], gps(Xd[ite, ])))))
sc$feature <- factor(sc$feature, levels = DESC[order(colMeans(abs(PHI)))])
print(ggplot(sc, aes(feature, m, fill = model)) +
        geom_col(position = position_dodge(width = .8), width = .75) + coord_flip() +
        scale_fill_manual(values = unname(PAL[c("blue", "orange")])) +
        labs(x = NULL, y = "mean |Shapley value| (log units)",
             title = "The scrambled model also has an explanation") +
        theme(legend.position = "bottom", legend.title = element_blank()))



<p class="caption"><b>Figure 7.</b> Mean absolute Shapley value per descriptor for the real model (blue) and for a model of identical architecture trained on permuted labels (orange), on the same feature ordering. The scrambled model has a held-out R² of −0.13 — worse than predicting the training mean — yet it distributes 61 % of the real model's total attribution mass across the descriptors, and its three top-ranked descriptors are drug-likeness, polar surface area and molecular weight: exactly the kind a reader would nod at.</p>

This is the single most useful figure to put in a supplementary section. The scrambled model's explanation is not noise-shaped; it is *story*-shaped, because a Shapley value faithfully reports what the model does, and what the model does is read the structure it over-fitted (its training \(R^2\) is 0.633 on random labels). An interpretation section that would survive this control states (i) the model's held-out performance, (ii) that the explanation was computed on held-out data, and (iii) ideally, the scrambled comparison.

**Control 2: stability across folds.** If the explanation is a property of the data and not of one particular fit, refitting on different subsets should preserve it.



In [ ]:
set.seed(0); fold <- sample(rep(1:5, length.out = nrow(Xd)))
FI <- t(sapply(1:5, function(k) {
  tr <- which(fold != k); te <- which(fold == k)
  set.seed(0); g <- xgb.train(PAR, xgb.DMatrix(Xd[tr, ], label = y[tr]), nrounds = 300, verbose = 0)
  perm_imp(function(X) predict(g, xgb.DMatrix(as.matrix(X))), Xd[te, ], y[te], nrep = 10)[, "mean"] }))
colnames(FI) <- DESC; rownames(FI) <- paste("fold", 1:5)
knitr::kable(round(FI, 3), caption = "Permutation importance refitted in five cross-validation folds")
pr <- combn(5, 2, function(ij) cor(FI[ij[1], ], FI[ij[2], ], method = "spearman"))
cat(sprintf("pairwise Spearman correlations between folds: min %.3f, median %.3f, max %.3f\n",
            min(pr), median(pr), max(pr)))
cat("top feature per fold:", paste(DESC[apply(FI, 1, which.max)], collapse = ", "), "\n")


In [ ]:
fl <- do.call(rbind, lapply(1:5, function(k) data.frame(fold = rownames(FI)[k], feature = DESC, imp = FI[k, ])))
fl$feature <- factor(fl$feature, levels = DESC[order(colMeans(FI))])
print(ggplot(fl, aes(feature, imp, colour = fold, group = fold)) +
        geom_point(size = 1.4) + geom_line(alpha = .4, linewidth = .3) + coord_flip() +
        scale_colour_viridis_d() +
        labs(x = NULL, y = "permutation importance (drop in R^2)",
             title = "The same explanation, five refits") +
        theme(legend.position = "right", legend.text = element_text(size = 6.5),
              legend.title = element_blank()))



<p class="caption"><b>Figure 8.</b> Permutation importance of the eleven descriptors in five cross-validation folds. The broad shape is reproduced — the same four descriptors occupy the top of every fold — but the ordering inside that group is not: the leading feature is fraction of sp3 carbon in two folds, rotatable bonds in two and hydrogen-bond acceptors in one, and the pairwise rank correlations run from 0.49 to 0.93.</p>

Note what that does and does not license. It licenses "the set of influential descriptors is reproducible under resampling"; it does not license "fraction of sp3 carbon drives ROCK1 potency", both because the rank of the top feature is itself unstable (median between-fold Spearman 0.73, not 0.95) and because Control 1 showed that a model with *no* predictive power produces a comparably confident ranking. Here the R edition is more pessimistic than the Python one, which found the same top feature in every fold with `shap` on LightGBM; the difference is the estimator, not the data — permutation importance on a 553-compound fold is noisier than a global TreeSHAP average.

**Control 3: the Rashomon effect.** Several models can fit the same data almost equally well and explain it differently — Breiman's "multiplicity of good models"<sup class="c"><a href="#r25">[25]</a></sup>. The practical test is to fit a few defensible alternatives and compare both their accuracy and their explanations.



In [ ]:
set.seed(0); rf <- ranger(x = as.data.frame(Xd[itr, ]), y = y[itr], num.trees = 300,
                          min.node.size = 2, seed = 1, num.threads = 4)
set.seed(0); et <- ranger(x = as.data.frame(Xd[itr, ]), y = y[itr], num.trees = 300, min.node.size = 2,
                          splitrule = "extratrees", num.random.splits = 5, seed = 1, num.threads = 4)
models <- list("ridge (linear)" = function(X) ridge_pred(scale(as.matrix(X), mu, sg)),
               "gradient boosting" = gp,
               "random forest" = function(X) predict(rf, data = as.data.frame(X))$predictions,
               "extra trees"   = function(X) predict(et, data = as.data.frame(X))$predictions)
IMP <- lapply(models, function(f) { p <- perm_imp(f, Xd[ite, ], y[ite], nrep = 10)[, "mean"]
                                    names(p) <- DESC; p })
rash <- do.call(rbind, lapply(names(models), function(nm) data.frame(
  model = nm, test_R2 = r2(y[ite], models[[nm]](Xd[ite, ])),
  top3 = paste(DESC[order(-IMP[[nm]])][1:3], collapse = ", "))))
knitr::kable(rash, digits = 3, row.names = FALSE, caption = "Four defensible models of the same table")
IM <- do.call(rbind, IMP)
prr <- combn(4, 2, function(ij) cor(IM[ij[1], ], IM[ij[2], ], method = "spearman"))
cat(sprintf("spread in test R2 across the four models: %.3f; across the three ensembles only: %.3f\n",
            diff(range(rash$test_R2)), diff(range(rash$test_R2[2:4]))))
cat(sprintf("pairwise Spearman between their importance profiles: min %.3f, median %.3f\n",
            min(prr), median(prr)))


In [ ]:
g1 <- ggplot(rash, aes(reorder(model, test_R2), test_R2, fill = model)) +
  geom_col(show.legend = FALSE) + coord_flip() + scale_fill_manual(values = unname(PAL)) +
  labs(x = NULL, y = "test R^2", title = "Four models, three of them comparable")
hm <- do.call(rbind, lapply(names(models), function(nm)
  data.frame(model = nm, feature = DESC, scaled = IMP[[nm]]/max(abs(IMP[[nm]])))))
hm$feature <- factor(hm$feature, levels = DESC)
g2 <- ggplot(hm, aes(feature, model, fill = scaled)) + geom_tile() +
  scale_fill_viridis_c(name = "importance\n(row-scaled)") +
  labs(x = NULL, y = NULL, title = "Permutation importance, each row scaled to its own maximum") +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 6.5),
        axis.text.y = element_text(size = 6.5), legend.key.height = unit(.35, "cm"),
        legend.title = element_text(size = 6))
print(g1 | g2)



<p class="caption"><b>Figure 9.</b> Left: held-out R² of four defensible models of the same data — the three tree ensembles within 0.022 of each other (0.323 to 0.345), ridge regression far below at 0.118. Right: their permutation-importance profiles, each row scaled to its own maximum. The three ensembles predict almost equally well and still disagree about the leading descriptor (gradient boosting and the random forest put fraction of sp3 carbon first, extra trees puts rotatable bonds first), and the pairwise rank correlations across all four profiles run from 0.06 to about 0.41.</p>

## 11. Explanations are not causal claims

Every method in this unit answers the same question: *how does the fitted function respond when I change its inputs?* None of them answers *what happens in a cell when a chemist changes a molecule*. Three gaps separate them.

**Confounding.** The training table is an accumulation of published series. If one laboratory's potent series happens to be lipophilic, the model will attribute potency to lipophilicity and the explanation will faithfully report that attribution. The association is real in the data and may be an artefact of publication history.

**Manipulability.** "Add three rotatable bonds" is not an intervention a chemist can perform without changing a dozen other descriptors at the same time; in the data those descriptors move together, as the 0.98 correlation of Section 2 shows. An explanation that varies one descriptor at a time describes a world in which they are separable.

**The noise floor.** Public IC50 and Ki values carry an experimental uncertainty of roughly 0.4–0.5 log units between laboratories<sup class="c"><a href="#r28">[28]</a></sup>. Most of the mean absolute Shapley values in Section 6.2 are below 0.18 log units — well inside the measurement error of the target they are explaining, and barely above the 0.12-log-unit Monte-Carlo error of the estimator itself. That is a decisive reason to report attributions with a scale, not as a bare ranking.

Rudin's conclusion<sup class="c"><a href="#r1">[1]</a></sup> follows from the three: for a decision that matters, prefer a model you can read to a black box with an explanation — and measure the cost of readability instead of assuming it. On this table the cost is large (ridge reaches R² 0.118 against 0.323 for boosting), so the ensemble is defensible; on many tabular problems it is not, and the comparison is one line of code. Either way the burden of proof belongs with the black box, and the controls of Section 10 are how it is discharged.

### How to write the interpretation paragraph

1. State the model and its held-out performance before any explanation.
2. Say which explanation method, on which data split, with which background distribution and how many samples, and give the magnitudes with units.
3. Distinguish global from local statements explicitly.
4. Report at least one control (scrambled labels, fold stability, or an alternative model class) and the agreement between two methods.
5. Write "the model used X" or "X is associated with Y in these data", never "X causes Y", unless you ran an experiment.

<div class="box bridge"><h4>Bridge</h4>
<p>The chemistry behind these explanations is taught in the cheminformatics course: fingerprints and descriptors in <a href="../../../Cheminformatics/U5_Descriptors_Fingerprints/index.html">U5</a>, similarity and SAR including activity cliffs in <a href="../../../Cheminformatics/U11_Similarity_SAR/index.html">U11</a>, classical QSAR and its interpretation in <a href="../../../Cheminformatics/U12_Classical_QSAR/index.html">U12</a>, and model validation in <a href="../../../Cheminformatics/U14_Validation/index.html">U14</a>. The ATP-site geometry that makes the hinge motif interpretable is covered in the bioinformatics course, <a href="../../../BGM324/03_Molecular_Docking/index.html">BGM324 unit 03 (molecular docking)</a> and <a href="../../../BGM324/04_Computer_Aided_Drug_Design/index.html">unit 04</a>. The algorithmic view of model evaluation is in the artificial-intelligence course, <a href="../../../BGM411/R/index.html">BGM411 (R edition)</a>. Inside this course, <a href="../U15_Clustering/index.html">Unit 15</a> clustered the same compounds without a target, <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> asks the complementary question — not "why this prediction" but "should this prediction be trusted at all" — and <a href="../U18_Deep_Learning/index.html">Unit 18</a> asks what a neural network would add. The Python edition of this unit, with TreeSHAP and atom-level colouring, is <a href="../../Python/U16_Interpretability/index.html">here</a>.</p></div>



In [ ]:
summary_tab <- data.frame(
  method = c("impurity importance", "permutation importance", "grouped permutation", "PDP / ICE", "ALE",
             "Shapley (exact enumeration)", "Shapley (iml, Monte-Carlo)", "DALEX break-down", "LIME",
             "counterfactual", "bit-level attribution"),
  scope = c("global", "global", "global", "global + per-instance", "global", "local", "local, aggregable",
            "local", "local", "local", "global, chemical"),
  result_here = c(
    sprintf("continuous noise ranked %d/13", which(names(imp) == "noise_continuous")),
    sprintf("top: %s (%.3f R²)", DESC[which.max(PI[, "mean"])], max(PI[, "mean"])),
    sprintf("size group %+.3f R² vs %+.3f alone", GP[4, "mean"], GP[1, "mean"]),
    sprintf("%s spans %.2f log units; %.0f%% of ICE curves rise", JF, diff(range(pdp$.value)), 100*rise),
    sprintf("%s spans %.2f log units", JF, diff(range(a1$ale))),
    sprintf("efficiency exact to %.0e", abs(sum(phi_exact) - (fx - Ef))),
    sprintf("max |MC - exact| %.3f at 200 samples", max(abs(phi_exact - phi_mc))),
    sprintf("rank agreement with Shapley %.2f", cor(phi0, bdv[DESC], method = "spearman")),
    sprintf("local R² %.2f; rank agreement with Shapley %.2f", L[[1]]$r2,
            cor(L[[1]]$coef, phi0, method = "spearman")),
    sprintf("cheapest change %.2f SD of %s", cf$change_in_SD[1], cf$descriptor[1]),
    sprintf("top bit in %.0f%% of library, %.0f%% potent", tab$pct_of_library[1], tab$pct_potent_when_set[1])),
  principal_failure_mode = c("biased to high-cardinality features", "masking + off-manifold inputs",
    "needs a grouping decided in advance", "extrapolates with correlated features",
    "bin choice; first order only", "exponential cost", "sampling error",
    "depends on the feature ordering", "sampling variance; manipulable",
    "may describe an impossible molecule", "bit cannot be inverted to a substructure in R"))
knitr::kable(summary_tab, row.names = FALSE)



<p class="caption"><b>Table 1.</b> The eleven explanations computed in this unit, with the number each produced and the failure mode that must be stated alongside it.</p>

## 12. Summary

Explanations are produced on demand and always look like knowledge, so they need the same discipline as predictions.

**Intrinsic first.** A linear model is its own explanation, exactly and permanently; everything else is a reconstruction. On these eleven descriptors the interpretable model is clearly worse (test R² 0.118 against 0.323 for gradient boosting), so the black box is justified here — but that is a measurement, not an assumption, and it is the measurement Rudin asks for<sup class="c"><a href="#r1">[1]</a></sup>.

**Global importances.** Impurity importance ranked a column of random numbers sixth of thirteen, above six real descriptors; never publish it. Permutation importance on held-out data is the model-agnostic replacement, but correlated features mask each other and the permutation itself creates impossible inputs<sup class="c"><a href="#r5">[5]</a></sup>; permuting the four size-related descriptors as a group cost 0.223 R² against 0.072 for the largest of them alone.

**Shapes.** PDP averages over feature combinations that may not exist, ICE shows whether the average means anything (73 % up, 27 % down here), ALE restricts the computation to intervals where data lives. The hand-written ALE and `iml`'s agreed to 0.003 log units; for molecular weight, PDP and ALE differ by 40 % in span.

**Attributions.** Shapley values are the unique attribution satisfying efficiency, symmetry, dummy and additivity<sup class="c"><a href="#r9">[9]</a><a href="#r10">[10]</a></sup>. Exact enumeration on a four-feature model satisfied efficiency to machine precision; the Monte-Carlo estimator reproduced the ranking but missed individual values by up to 0.11 log units, and `DALEX`'s break-down — a different decomposition, not a Shapley value — agreed with it only at rank correlation 0.76. LIME gave a local linear story whose surrogate explained about 40 % of the black box's local variance and whose ranking correlated with Shapley's at 0.11.

**Domain mapping.** The bit-to-atom inversion that lets the Python edition colour molecules needs RDKit and cannot be done here; what the frozen fingerprints do support is the layer underneath — the ten bits the model leans on most are carried by 11–52 % of the library and their carriers are up to 1.9 times as likely to be potent as the library average.

**Controls.** A y-scrambled model with a held-out R² of −0.13 produced 61 % of the real model's attribution mass, with a chemically plausible top three. Fold-to-fold rank correlations were 0.49 to 0.93, so the influential *set* is reproducible and its *order* is not. Three tree ensembles within 0.022 R² of each other disagreed about which descriptor matters most, their importance profiles correlating as low as 0.06.

Everything in this unit runs in R<sup class="c"><a href="#r34">[34]</a></sup> with open packages — `ranger`, `xgboost`, `iml`, `DALEX`, `glmnet` — from the public ROCK1 table, in under two minutes on a laptop CPU.

**Causality.** Every explanation here is a statement about the fitted function. Confounding by publication history, the impossibility of varying one descriptor alone, and an experimental noise floor of about half a log unit<sup class="c"><a href="#r28">[28]</a></sup> each separate "the model used X" from "X causes potency". <a href="../U17_Uncertainty_AD/index.html">Unit 17</a> takes the next step and asks whether a given prediction should be made at all.

## 13. Self-assessment

<details><summary>1. Why is impurity importance biased, and in which direction?</summary>
<p>Because it accumulates the criterion reduction achieved by whichever splits the tree happened to choose, and a feature with many candidate thresholds (continuous or high-cardinality) has more chances to fit the training residuals than a binary one. The bias favours continuous and many-valued features: in Section 3 a column of standard normal noise took 9.7 % of the total importance and ranked sixth of thirteen while a random binary column ranked last with 1.8 %.</p></details>

<details><summary>2. Two descriptors correlate at 0.98 and both show small permutation importance. What do you conclude?</summary>
<p>Nothing about their relevance. Destroying one leaves the model able to read the other, so each appears partly dispensable — masking. Permute them together: in Section 4 the pair cost 0.084 R² against 0.072 and 0.025 individually, and the four-descriptor size group cost 0.223, more than twice the sum of its parts.</p></details>

<details><summary>3. When is a partial-dependence plot misleading, and what replaces it?</summary>
<p>When the feature is correlated with others, because the PDP evaluates the model at combinations that do not occur (a molecule with the weight of a peptide and the heavy-atom count of benzene). ALE replaces it: it accumulates average prediction differences computed only inside intervals where data exists. For molecular weight — correlated at 0.98 with heavy-atom count — the PDP spanned 0.458 log units and ALE 0.651. Also plot ICE curves to see whether the average hides opposing individual effects.</p></details>

<details><summary>4. State the four axioms that make Shapley values unique, and which one lets you draw a waterfall plot.</summary>
<p>Efficiency, symmetry, dummy and additivity. Efficiency — the attributions sum to the prediction minus the expected prediction — is what makes a waterfall add up exactly, as verified numerically in Section 6.1, where the exact enumeration matched f(x) − E[f] to machine precision.</p></details>

<details><summary>5. You use <code>iml::Shapley</code> instead of TreeSHAP. What must you report, and why?</summary>
<p>The number of Monte-Carlo samples and the efficiency gap. <code>iml::Shapley</code> estimates the same quantity by sampling feature orderings, so its output is noisy: with 200 samples the individual values differed from the exact enumeration by up to 0.11 log units, and with 60 samples the attributions missed the prediction by about 0.12 log units on average. Since many of the mean attributions themselves are below 0.18, the estimator's error is of the same order as the effects being compared.</p></details>

<details><summary>6. Your LIME explanation changes when you change the seed. Is the method broken?</summary>
<p>No — it is sampling-based, and the variability is part of its output. Report several seeds and the local R² of the surrogate; here 2,000 perturbations gave rank correlations of 0.98 and 0.99 between seeds, so the instability is modest, but the surrogate reproduced only about 40 % of the black box's local variance and ranked the descriptors almost independently of the Shapley values (Spearman 0.11). Because the perturbations are drawn off the data manifold, they can in principle be exploited to hide a model's behaviour.</p></details>

<details><summary>7. A counterfactual says "raise the number of rotatable bonds from 6 to 9 and the compound becomes active". What are the two objections?</summary>
<p>First, descriptor-space feasibility: no molecule may exist with that descriptor vector, and nothing checks synthetic accessibility — the same table also offers "raise the molecular weight to 823 Da". Second, non-manipulability: rotatable bonds cannot be changed alone, since any real modification moves several descriptors together, so the counterfactual describes a direction in feature space, not a chemical plan.</p></details>

<details><summary>8. How would you turn a 2,048-bit fingerprint explanation into something a chemist can read, and what is possible in R here?</summary>
<p>With a toolkit: each set bit records a circular atom environment with a known central atom and radius, so each bit's attribution is distributed over the atoms of its environment and drawn on the molecule. In this environment the hash cannot be inverted, so the unit stops one level lower — which bits the model uses, how many compounds carry them, and how potency is distributed among the carriers. State the fingerprint, the radius, and which of the two you did.</p></details>

<details><summary>9. A y-scrambled model's attribution summary looks chemically sensible. What does that prove?</summary>
<p>That "the explanation looks sensible" carries no information about whether the model learned anything. The scrambled model in Section 10 had a held-out R² of −0.13 and still carried 61 % of the real model's attribution mass, with drug-likeness, polar surface area and molecular weight on top. An interpretation section must therefore lead with held-out performance.</p></details>

<details><summary>10. Fold-to-fold Spearman correlations of permutation importance range from 0.49 to 0.93. What may you claim?</summary>
<p>That the same small group of descriptors is influential in every fold — the set is reproducible — but not that any particular one leads, because the top feature changed between folds. And not that the ranking is causal, nor that another model class would agree: Figure 9 shows it does not.</p></details>

<details><summary>11. What is the Rashomon effect and how should it change what you write?</summary>
<p>Several models can fit the data almost equally well while attributing the signal to different features: in Section 10 the three tree ensembles were within 0.022 R² of each other and still disagreed about the leading descriptor, with importance profiles correlating as low as 0.06. It implies that "the important features are A, B, C" is a statement about one fitted model, not about the problem; write it that way, and report at least one alternative model class.</p></details>

<details><summary>12. Why does the experimental error of public bioactivity data limit what an explanation can mean?</summary>
<p>Because the target itself is uncertain by roughly half a log unit between laboratories, any attribution smaller than that is inside the noise of what is being explained. Here the largest mean attribution was 0.179 log units — a third of the experimental floor. Report attributions with units and compare them with the measurement error before interpreting a ranking position.</p></details>

## Glossary

<dl class="gloss">
<dt>ALE (accumulated local effects)</dt><dd>Effect curve built by accumulating average prediction differences across bin edges, using only data inside each bin; unbiased under correlated features.</dd>
<dt>Break-down (DALEX)</dt><dd>Sequential decomposition of a prediction along one greedy ordering of features; sums exactly to the prediction but is not a Shapley value.</dd>
<dt>Counterfactual explanation</dt><dd>The smallest change to an instance's features that flips the model's decision.</dd>
<dt>Efficiency gap</dt><dd>Difference between the sum of estimated attributions and the prediction minus its expectation; zero for exact Shapley values, non-zero for sampled ones.</dd>
<dt>Global vs local explanation</dt><dd>A statement about the model's average behaviour over a population versus about one individual prediction.</dd>
<dt>ICE (individual conditional expectation)</dt><dd>One partial-dependence curve per observation, plotted without averaging.</dd>
<dt>Impurity importance</dt><dd>Total reduction in the splitting criterion attributed to a feature in a tree ensemble; biased towards features with many split points.</dd>
<dt>Interventional vs conditional value function</dt><dd>In Shapley estimation, replacing absent features by independent background draws versus by draws from their conditional distribution.</dd>
<dt>Intrinsically interpretable model</dt><dd>A model whose structure is itself the explanation: linear model, shallow tree, rule list, generalised additive model.</dd>
<dt>LIME</dt><dd>Local surrogate: a linear model fitted to the black box's predictions on perturbations around one instance, weighted by proximity.</dd>
<dt>Masking</dt><dd>Mutual suppression of permutation importance between correlated features, each covering for the other.</dd>
<dt>Partial dependence (PDP)</dt><dd>Average prediction when one feature is set to a value for every observation and the others are left unchanged.</dd>
<dt>Permutation importance</dt><dd>Drop in a held-out performance metric when one feature's values are shuffled.</dd>
<dt>Rashomon effect</dt><dd>The existence of several models with nearly equal accuracy and materially different explanations.</dd>
<dt>Shapley value</dt><dd>Average marginal contribution of a feature over all coalitions; the unique attribution satisfying efficiency, symmetry, dummy and additivity.</dd>
<dt>TreeSHAP</dt><dd>Polynomial-time exact algorithm for Shapley values of tree ensembles; available in Python, not in this R environment.</dd>
<dt>y-scrambling</dt><dd>Refitting the identical pipeline on permuted labels as a negative control, for performance and for explanations alike.</dd>
</dl>

## References and further reading

<ol class="refs">
<li id="r1">Rudin C. Stop explaining black box machine learning models for high stakes decisions and use interpretable models instead. <em>Nat Mach Intell</em>. 2019;1(5):206-215. <a href="https://doi.org/10.1038/s42256-019-0048-x">doi:10.1038/s42256-019-0048-x</a></li>
<li id="r2">Lipton ZC. The mythos of model interpretability. <em>Commun ACM</em>. 2018;61(10):36-43. <a href="https://doi.org/10.1145/3233231">doi:10.1145/3233231</a></li>
<li id="r3">Breiman L. Random forests. <em>Mach Learn</em>. 2001;45:5-32. <a href="https://doi.org/10.1023/A:1010933404324">doi:10.1023/A:1010933404324</a></li>
<li id="r4">Strobl C, Boulesteix AL, Zeileis A, Hothorn T. Bias in random forest variable importance measures: illustrations, sources and a solution. <em>BMC Bioinformatics</em>. 2007;8:25. <a href="https://doi.org/10.1186/1471-2105-8-25">doi:10.1186/1471-2105-8-25</a></li>
<li id="r5">Hooker G, Mentch L, Zhou S. Unrestricted permutation forces extrapolation: variable importance requires at least one more model, or there is no free variable importance. <em>Stat Comput</em>. 2021;31:82. <a href="https://doi.org/10.1007/s11222-021-10057-z">doi:10.1007/s11222-021-10057-z</a></li>
<li id="r6">Friedman JH. Greedy function approximation: a gradient boosting machine. <em>Ann Stat</em>. 2001;29(5):1189-1232. <a href="https://doi.org/10.1214/aos/1013203451">doi:10.1214/aos/1013203451</a></li>
<li id="r7">Goldstein A, Kapelner A, Bleich J, Pitkin E. Peeking inside the black box: visualizing statistical learning with plots of individual conditional expectation. <em>J Comput Graph Stat</em>. 2015;24(1):44-65. <a href="https://doi.org/10.1080/10618600.2014.907095">doi:10.1080/10618600.2014.907095</a></li>
<li id="r8">Apley DW, Zhu J. Visualizing the effects of predictor variables in black box supervised learning models. <em>J R Stat Soc Series B</em>. 2020;82(4):1059-1086. <a href="https://doi.org/10.1111/rssb.12377">doi:10.1111/rssb.12377</a></li>
<li id="r9">Shapley LS. A value for n-person games. In: Kuhn HW, Tucker AW, eds. <em>Contributions to the Theory of Games II</em>. Princeton University Press; 1953:307-317. <a href="https://doi.org/10.1515/9781400881970-018">doi:10.1515/9781400881970-018</a></li>
<li id="r10">Lundberg SM, Lee SI. A unified approach to interpreting model predictions. In: <em>NeurIPS 30</em>; 2017. <a href="https://doi.org/10.48550/arXiv.1705.07874">doi:10.48550/arXiv.1705.07874</a></li>
<li id="r11">Lundberg SM, Erion G, Chen H, et al. From local explanations to global understanding with explainable AI for trees. <em>Nat Mach Intell</em>. 2020;2(1):56-67. <a href="https://doi.org/10.1038/s42256-019-0138-9">doi:10.1038/s42256-019-0138-9</a></li>
<li id="r12">Aas K, Jullum M, Loland A. Explaining individual predictions when features are dependent: more accurate approximations to Shapley values. <em>Artif Intell</em>. 2021;298:103502. <a href="https://doi.org/10.1016/j.artint.2021.103502">doi:10.1016/j.artint.2021.103502</a></li>
<li id="r13">Ribeiro MT, Singh S, Guestrin C. "Why should I trust you?": explaining the predictions of any classifier. In: <em>Proc. 22nd ACM SIGKDD</em>; 2016:1135-1144. <a href="https://doi.org/10.1145/2939672.2939778">doi:10.1145/2939672.2939778</a></li>
<li id="r14">Slack D, Hilgard S, Jia E, Singh S, Lakkaraju H. Fooling LIME and SHAP: adversarial attacks on post hoc explanation methods. In: <em>AAAI/ACM Conf. on AI, Ethics, and Society</em>; 2020:180-186. <a href="https://doi.org/10.1145/3375627.3375830">doi:10.1145/3375627.3375830</a></li>
<li id="r15">Rodriguez-Perez R, Bajorath J. Interpretation of compound activity predictions from complex machine learning models using local approximations and Shapley values. <em>J Med Chem</em>. 2020;63(16):8761-8777. <a href="https://doi.org/10.1021/acs.jmedchem.9b01101">doi:10.1021/acs.jmedchem.9b01101</a></li>
<li id="r16">Jimenez-Luna J, Grisoni F, Schneider G. Drug discovery with explainable artificial intelligence. <em>Nat Mach Intell</em>. 2020;2(10):573-584. <a href="https://doi.org/10.1038/s42256-020-00236-4">doi:10.1038/s42256-020-00236-4</a></li>
<li id="r17">Mothilal RK, Sharma A, Tan C. Explaining machine learning classifiers through diverse counterfactual explanations. In: <em>Proc. 2020 Conf. on Fairness, Accountability, and Transparency</em>; 2020:607-617. <a href="https://doi.org/10.1145/3351095.3372850">doi:10.1145/3351095.3372850</a></li>
<li id="r18">Rogers D, Hahn M. Extended-connectivity fingerprints. <em>J Chem Inf Model</em>. 2010;50(5):742-754. <a href="https://doi.org/10.1021/ci100050t">doi:10.1021/ci100050t</a></li>
<li id="r19">Stumpfe D, Bajorath J. Exploring activity cliffs in medicinal chemistry. <em>J Med Chem</em>. 2012;55(7):2932-2942. <a href="https://doi.org/10.1021/jm201706b">doi:10.1021/jm201706b</a></li>
<li id="r20">van Tilborg D, Alenicheva A, Grisoni F. Exposing the limitations of molecular machine learning with activity cliffs. <em>J Chem Inf Model</em>. 2022;62(23):5938-5951. <a href="https://doi.org/10.1021/acs.jcim.2c01073">doi:10.1021/acs.jcim.2c01073</a></li>
<li id="r21">Riniker S, Landrum GA. Similarity maps - a visualization strategy for molecular fingerprints and machine-learning methods. <em>J Cheminform</em>. 2013;5:43. <a href="https://doi.org/10.1186/1758-2946-5-43">doi:10.1186/1758-2946-5-43</a></li>
<li id="r22">Feng Y, LoGrasso PV, Defert O, Li R. Rho kinase (ROCK) inhibitors and their therapeutic potential. <em>J Med Chem</em>. 2016;59(6):2269-2300. <a href="https://doi.org/10.1021/acs.jmedchem.5b00683">doi:10.1021/acs.jmedchem.5b00683</a></li>
<li id="r23">Rucker C, Rucker G, Meringer M. y-Randomization and its variants in QSPR/QSAR. <em>J Chem Inf Model</em>. 2007;47(6):2345-2357. <a href="https://doi.org/10.1021/ci700157b">doi:10.1021/ci700157b</a></li>
<li id="r24">Zdrazil B, Felix E, Hunter F, et al. The ChEMBL Database in 2023. <em>Nucleic Acids Res</em>. 2024;52(D1):D1180-D1192. <a href="https://doi.org/10.1093/nar/gkad1004">doi:10.1093/nar/gkad1004</a></li>
<li id="r25">Breiman L. Statistical modeling: the two cultures. <em>Stat Sci</em>. 2001;16(3):199-231. <a href="https://doi.org/10.1214/ss/1009213726">doi:10.1214/ss/1009213726</a></li>
<li id="r26">Molnar C, Konig G, Herbinger J, et al. General pitfalls of model-agnostic interpretation methods for machine learning models. In: <em>xxAI - Beyond Explainable AI</em>. LNCS 13200. Springer; 2022:39-68. <a href="https://doi.org/10.1007/978-3-031-04083-2_4">doi:10.1007/978-3-031-04083-2_4</a></li>
<li id="r27">Polishchuk P. Interpretation of quantitative structure-activity relationship models: past, present, and future. <em>J Chem Inf Model</em>. 2017;57(11):2618-2639. <a href="https://doi.org/10.1021/acs.jcim.7b00274">doi:10.1021/acs.jcim.7b00274</a></li>
<li id="r28">Kramer C, Kalliokoski T, Gedeck P, Vulpetti A. The experimental uncertainty of heterogeneous public Ki data. <em>J Med Chem</em>. 2012;55(11):5165-5173. <a href="https://doi.org/10.1021/jm300131x">doi:10.1021/jm300131x</a></li>
<li id="r29">Molnar C, Bischl B, Casalicchio G. iml: an R package for interpretable machine learning. <em>J Open Source Softw</em>. 2018;3(26):786. <a href="https://doi.org/10.21105/joss.00786">doi:10.21105/joss.00786</a></li>
<li id="r30">Wright MN, Ziegler A. ranger: a fast implementation of random forests for high dimensional data in C++ and R. <em>J Stat Softw</em>. 2017;77(1):1-17. <a href="https://doi.org/10.18637/jss.v077.i01">doi:10.18637/jss.v077.i01</a></li>
<li id="r31">Friedman J, Hastie T, Tibshirani R. Regularization paths for generalized linear models via coordinate descent. <em>J Stat Softw</em>. 2010;33(1):1-22. <a href="https://doi.org/10.18637/jss.v033.i01">doi:10.18637/jss.v033.i01</a></li>
<li id="r32">Biecek P. DALEX: explainers for complex predictive models in R. <em>J Mach Learn Res</em>. 2018;19(84):1-5. <a href="https://doi.org/10.48550/arXiv.1806.08915">doi:10.48550/arXiv.1806.08915</a></li>
<li id="r33">Chen T, Guestrin C. XGBoost: a scalable tree boosting system. In: <em>Proc. 22nd ACM SIGKDD</em>; 2016:785-794. <a href="https://doi.org/10.1145/2939672.2939785">doi:10.1145/2939672.2939785</a></li>
<li id="r34">Ihaka R, Gentleman R. R: a language for data analysis and graphics. <em>J Comput Graph Stat</em>. 1996;5(3):299-314. <a href="https://doi.org/10.1080/10618600.1996.10474713">doi:10.1080/10618600.1996.10474713</a></li>
<li id="r35">Wickham H. <em>ggplot2: Elegant Graphics for Data Analysis</em>. 2nd ed. Springer; 2016. <a href="https://doi.org/10.1007/978-3-319-24277-4">doi:10.1007/978-3-319-24277-4</a></li>
</ol>

<h4>Textbooks</h4>
<ul>
<li>Molnar C. <em>Interpretable Machine Learning</em>. 2nd ed.; 2022 (free online). - The reference for every method in this unit; the author also wrote the <code>iml</code> package<sup class="c"><a href="#r29">[29]</a></sup>.</li>
<li>Biecek P, Burzykowski T. <em>Explanatory Model Analysis</em>. Chapman &amp; Hall; 2021 (free online). - The book behind <code>DALEX</code>, with R code for every figure in this unit.</li>
<li>Hastie T, Tibshirani R, Friedman J. <em>The Elements of Statistical Learning</em>. 2nd ed. Springer; 2009. - Section 10.13 on partial dependence and relative importance.</li>
<li>Wickham H. <em>ggplot2</em><sup class="c"><a href="#r35">[35]</a></sup> and Wickham H, Cetinkaya-Rundel M, Grolemund G. <em>R for Data Science</em>. 2nd ed. O'Reilly; 2023. - The plotting and data idioms used throughout.</li>
</ul>

<h4>Online resources</h4>
<ul>
<li><code>iml</code> package documentation and vignettes: <a href="https://cran.r-project.org/package=iml">cran.r-project.org/package=iml</a>.</li>
<li><code>DALEX</code> and the DrWhy.AI ecosystem: <a href="https://modeloriented.github.io/DALEX/">modeloriented.github.io/DALEX</a>.</li>
<li><code>fastshap</code> and <code>treeshap</code>, the R packages that approximate or port TreeSHAP (not installed here): <a href="https://cran.r-project.org/package=fastshap">cran.r-project.org/package=fastshap</a>.</li>
<li>Practical Cheminformatics (P. Walters), posts on interpreting QSAR models: <a href="https://patwalters.github.io/practicalcheminformatics/">patwalters.github.io/practicalcheminformatics</a>.</li>
</ul>
